# 05. Agent 통합 테스트

04에서 단독 검증된 도구 3개(guide·weather·currency)를 `create_react_agent`로 묶고, 시연 시나리오 5개를 실제 Agent로 돌려본다.

**검증 포인트**
1. Agent가 질문 종류에 따라 적절한 도구를 자동 선택하는가
2. 03_rag_test에서 잡지 못한 시점·수치 환각이 도구 통합 후 자연 해소되는가
3. 페르소나 가드가 작동해 사이판 정보가 답변에 섞이지 않는가 (plan 위험 신호 표)
4. MemorySaver + thread_id로 후속 질문에서 이전 맥락을 기억하는가 (시연 #5)
5. 환율 환산 계산이 LLM 쪽에서 정확히 이루어지는가 (옵션 A 분리 검증)

**시연 시나리오 5개**
1. 순수 RAG: "PIC 리조트 어때? 가족이 가도 좋아?"
2. 도구 1개 — 날씨: "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"
3. 도구 1개 — 환율: "100달러면 한국 돈 얼마야?"
4. 멀티 도구 (RAG + 날씨 + 환율, 3개 멀티): "5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘"
5. 메모리: 시연 #4 후속 — "그럼 다른 추천도 있어?"

## 1. 환경 설정 + Agent 객체 import

In [ ]:
import sys
from pathlib import Path

# 노트북 작업 디렉토리는 practice/, 그 한 단계 위 src/를 path에 추가
sys.path.insert(0, str(Path.cwd().parent / "src"))

from langchain_core.messages import AIMessage, ToolMessage

from guam_chatbot.agent import agent, invoke_agent, TOOL_NAMES

print(f"Agent 타입: {type(agent).__name__}")
print(f"Agent 노드: {list(agent.nodes.keys())}")
print(f"\n등록된 도구:")
for name in TOOL_NAMES:
    print(f"  - {name}")

## 헬퍼 — 시연 결과 출력 포맷

각 시연마다 응답 시간, 도구 호출 흐름, 최종 답변을 일관되게 보여주도록 헬퍼 정의.

In [ ]:
import time

def run_demo(question: str, thread_id: str, label: str, verbose: bool = False):
    """시연 시나리오 1건 실행 + 결과 정리 출력.
    
    verbose=True면 ToolMessage 본문(LLM이 실제로 받은 raw 도구 결과)까지 출력.
    환각 진단용: AIMessage 답변에 있는 정보가 ToolMessage에 정말 있는지 직접 확인.
    """
    print("=" * 70)
    print(f"[{label}] thread_id={thread_id}")
    print(f"Q: {question}")
    print("=" * 70)

    start = time.time()
    result = invoke_agent(question, thread_id=thread_id)
    elapsed = time.time() - start

    msgs = result["messages"]
    tool_calls = [m for m in msgs if isinstance(m, ToolMessage)]

    print(f"\n⏱️  응답 시간: {elapsed:.2f}초")
    print(f"📞 도구 호출 횟수: {len(tool_calls)}")
    for tm in tool_calls:
        print(f"   - {tm.name}")

    if verbose:
        print(f"\n=== 메시지 흐름 (환각 진단용) ===")
        for i, m in enumerate(msgs):
            kind = type(m).__name__
            if isinstance(m, ToolMessage):
                print(f"\n--- [{i}] {kind} (tool={m.name}) ---")
                print(m.content)  # raw 도구 결과 — 길이 제한 없음
            elif isinstance(m, AIMessage):
                if m.tool_calls:
                    print(f"\n--- [{i}] {kind} (도구 호출 결정) ---")
                    for tc in m.tool_calls:
                        print(f"  → {tc['name']}({tc.get('args', {})})")
                else:
                    print(f"\n--- [{i}] {kind} (최종 답변) ---")
                    print(m.content)
            else:
                print(f"\n--- [{i}] {kind} ---")
                print(m.content[:300])
    else:
        print(f"\n=== 최종 답변 ===")
        print(msgs[-1].content)
    print()
    return result

## 2. 시연 #1 — 순수 RAG

Q: "PIC 리조트 어때? 가족이 가도 좋아?"

**기대 동작**: search_guam_guide 1회 호출 → PIC 정보 기반 답변 + 출처 인용

**확인할 것**
- search_guam_guide만 호출되는지 (날씨·환율 도구 안 부르는지)
- 답변에 "PIC"·"가족"·"한국어 직원"·"수영장" 등 핵심 정보 포함
- 출처 인용 (예: `[리조트] 퍼시픽 아일랜드 클럽 - 나무위키`)
- ⚠️ **사이판 PIC 정보가 답변에 섞이지 않는지** (페르소나 가드 검증)

In [ ]:
_ = run_demo(
    question="PIC 리조트 어때? 가족이 가도 좋아?",
    thread_id="demo1",
    label="시연 #1 — 순수 RAG",
)

## 3. 시연 #2 — 도구 1개 (날씨)

Q: "5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?"

**기대 동작**: get_guam_weather 1회 호출 → 5일치 일별 데이터 기반 답변

**확인할 것**
- get_guam_weather만 호출되는지
- 03_rag_test의 시점·수치 환각("70~100mm" 등)이 사라졌는지 → **3섹션 RAG 한계 보강 검증 포인트**
- 비 확률 100% 일자가 답변에 반영되어 "우비 챙기세요" 식 결론 도출
- weather.py의 어색한 한국어 description("실 비"/"온흐림")이 LLM이 자연스럽게 풀어주는지

In [ ]:
_ = run_demo(
    question="5월 둘째 주 괌 날씨 어때? 우비 챙겨야 해?",
    thread_id="demo2",
    label="시연 #2 — 날씨",
)

## 4. 시연 #3 — 도구 1개 (환율)

Q: "100달러면 한국 돈 얼마야?"

**기대 동작**: convert_currency 1회 호출 → 환율값 받아서 LLM이 100×환율 계산해 답변

**확인할 것**
- convert_currency만 호출되는지
- 옵션 A 분리 검증: 도구는 환율만, 환산(`100 × 1450.80 = 145,080원`)은 LLM이 직접
- 답변에 환산 계산 과정 또는 결과 포함 (대략 14만~15만원 수준)
- 답변에 영업일·매매기준율 안내 자연스럽게 녹여내는지

In [ ]:
_ = run_demo(
    question="100달러면 한국 돈 얼마야?",
    thread_id="demo3",
    label="시연 #3 — 환율",
)

## 5. 시연 #4 — 멀티 도구 (RAG + 날씨 + 환율, 3개 멀티)

Q: "5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘"

**기대 동작**: search_guam_guide + get_guam_weather + convert_currency 3개 모두 호출 → PIC 정보 + 5월 둘째 주 일별 예보 + USD/KRW 환율 통합 답변

**확인할 것**
- 도구 3개 모두 호출되는지 (Agent의 도구 선택 로직 검증)
- 100달러 × 환율 환산 계산이 답변에 들어가는지 (옵션 A 분리)
- PIC 가격 정보가 RAG에 없으면 "공식 사이트에서 직접 확인" 가드 멘트로 마무리되는지
- thread_id `demo4` — 시연 #5에서 이어서 사용

In [ ]:
demo4_result = run_demo(
    question="5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘",
    thread_id="demo4",
    label="시연 #4 — 멀티 도구 (RAG + 날씨 + 환율)",
    verbose=True,  # ← 추가
)

## 6. 시연 #5 — 메모리 (시연 #4 후속)

Q: "그럼 다른 추천도 있어?"

**기대 동작**: thread_id `demo4`로 다시 호출 → MemorySaver가 #4 맥락 자동 로드 → "PIC 외 다른 가족 친화 추천"으로 해석

**확인할 것**
- thread_id `demo4`가 #4와 동일 — 메모리 연결
- "그럼"이 무엇을 가리키는지 LLM이 #4 맥락에서 판단 (PIC + 5월 둘째 주 + 가족 5인 맥락)
- search_guam_guide 다시 호출 (다른 리조트·명소 정보 찾기)
- ⚠️ 자료에 없는 호텔명·가격이 등장하면 환각 회귀 (RAG 한계 — Plan B 멘트 대비 필요)

In [ ]:
_ = run_demo(
    question="그럼 다른 추천도 있어?",
    thread_id="demo4",
    label="시연 #5 — 메모리 (#4 후속)",
    verbose=True,  # ← 추가
)

## 7. 검증 결과

| 시연 | 응답 시간 (참고치) | 도구 호출 | 핵심 검증 |
|---|---|---|---|
| #1 순수 RAG | ~3초 | search_guam_guide 1회 | PIC 정보 + 출처, 사이판 혼입 0건 |
| #2 날씨 | ~3초 | get_guam_weather 1회 | 5일 일별 예보, 우비 결론 도출 |
| #3 환율 | ~2초 | convert_currency 1회 | 옵션 A 분리, LLM이 100×환율 환산 |
| #4 멀티 | ~5초 | RAG + 날씨 + 환율 3개 | 3개 도구 동시 호출, 가격 환각 가드 멘트로 마무리 |
| #5 메모리 | ~4초 | search_guam_guide 재호출 | #4 맥락에서 "그럼" 해석, ⚠️ 호출 변동 잔존 (Plan B 대비) |

**한계 인지** (강의 S6_1 RAG 한계와 일치)
- temperature=0인데도 호출마다 답변 변동 폭이 큼
- 시연 #4·#5는 호출 회귀 시 환각 잔존 가능성. 발표 라이브 시 Plan B 멘트로 RAG 한계·시나리오 설계 중요성 설명 가능
- 가격·노후화·우회 표현 등 가드(SYSTEM_PROMPT 규칙 4번)가 일부 우회 사례 있음

In [ ]:
# ===== 환각 빈도 측정 (시연 #4 5회 회귀) =====
# 환각 정의: 답변에 자료에 없는 가격 견적/호텔명 등장
# 매 호출 새 thread_id → 이전 호출 메모리 영향 배제

import re
import uuid

QUERY = "5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘"
N = 5

# 환각 패턴 (측정 결과 기반)
HALLUCINATION_PATTERNS = [
    (r'\d+\s*~\s*\d+\s*만\s*원',     '가격 견적 범위 (X~Y만 원)'),
    (r'약\s*\d+\s*만\s*원',          '약 ~만원 우회 표현'),
    (r'평균\s*\d+\s*(만\s*원|달러)', '평균 ~만원/달러 우회 표현'),
    (r'추정\s*\d+',                  '추정 ~ 우회 표현'),
    (r'대략\s*\d+',                  '대략 ~ 우회 표현'),
    (r'하몬\s*인|Harmon\s*Inn',      '환각 호텔명: 하몬 인'),
    (r'로열\s*라구나|Royal\s*Laguna','환각 호텔명: 로열 라구나'),
    (r'호텔스닷컴|아고다|익스피디아', '환각 사이트명'),
]

results = []
for i in range(N):
    print(f"\n{'='*60}\n호출 {i+1}/{N}\n{'='*60}")
    
    # 새 thread_id (이전 호출 메모리 영향 배제)
    thread_id = f"measure-{uuid.uuid4().hex[:8]}"
    
    result = invoke_agent(QUERY, thread_id=thread_id)
    answer = result["messages"][-1].content
    
    # 자동 환각 검출 (보조 — 최종 판정은 사용자 눈으로)
    detected = []
    for pattern, label in HALLUCINATION_PATTERNS:
        matches = re.findall(pattern, answer)
        if matches:
            detected.append((label, matches))
    
    is_halluc = len(detected) > 0
    results.append({"call": i+1, "answer": answer, "auto_halluc": is_halluc, "detected": detected})
    
    print(f"\n[답변]\n{answer}\n")
    print(f"[자동 검출] {'⚠️ 환각 의심' if is_halluc else '✓ 깨끗 (자동 패턴 기준)'}")
    for label, matches in detected:
        print(f"  - {label}: {matches}")

# 집계
print(f"\n{'='*60}")
print(f"자동 검출 기준 환각 빈도: {sum(r['auto_halluc'] for r in results)}/{N}")
print(f"{'='*60}")
print("\n⚠️ 자동 검출은 보조. 최종 판정은 위 5개 답변 직접 읽고 판단하세요:")
print("  - 가격 견적이 자료에 없는데 등장하는가?")
print("  - 호텔명이 자료에 없는데 등장하는가?")
print("  - '약/평균/추정/대략' 같은 우회 표현이 등장하는가?")

In [ ]:
# ===== 시연 #4 + #5 메모리 흐름 1회 동작 확인 =====
# 같은 thread_id로 #4 → #5 연속 호출. #5의 "그럼"이 #4 맥락(PIC, 5월 둘째 주, 100달러)을 잇는지.

thread_id = "demo45"

print("=" * 60)
print("[시연 #4 (재설계)]")
print("=" * 60)
q4 = "5인 가족이 5월 둘째 주에 괌 갈 건데, PIC 어때? 날씨도 보고 싶고 100달러 환전 환율도 알려줘"
r4 = invoke_agent(q4, thread_id=thread_id)
print(r4["messages"][-1].content)

print("\n" + "=" * 60)
print("[시연 #5 (메모리)]")
print("=" * 60)
q5 = "그럼 다른 추천도 있어?"
r5 = invoke_agent(q5, thread_id=thread_id)
print(r5["messages"][-1].content)